# Rag Assignment
Authors: Gianluca Amato and David Farrugia

--- short notebook description and overview of sections

## 1. Setup

### 1.1 Imports

This section imports all the required Python libraries used throughout the notebook and a fixed random seed is set for both Python’s `random` module and NumPy for reproducible results across multiple executions of the notebook.

In [ ]:
import time
import random
import math
from pathlib import Path

import numpy as np
import pandas as pd

import plotly.express as px
from IPython.display import display

import requests
import faiss
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from sentence_transformers import SentenceTransformer
from rouge_score import rouge_scorer
from sacrebleu import corpus_bleu
from sklearn.metrics.pairwise import cosine_similarity
from transformers import (DPRQuestionEncoder, 
                          DPRQuestionEncoderTokenizer, 
                          DPRContextEncoder, 
                          DPRContextEncoderTokenizer, 
                          AutoTokenizer, 
                          AutoModelForSeq2SeqLM)

# Download necessary NLTK resources
nltk.download('stopwords')
nltk.download('punkt_tab')

# Set random seed for reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\david\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\david\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


### 1.2 Path Setup and Input Validation

This section is responsible for setting up the directory structure required by the notebook and validating the presence of essential input data files.

Output and temporary dataset directories are created automatically if they do not already exist, ensuring that files can be written without manual setup at all stages of the notebook. The use of `Pathlib` allows for clean and platform independent path management.

The input dataset directory and the expected CSV file are then defined. A helper function is initialised to validate the input CSV path by checking that the file exists, is a valid file (not a directory), and has the correct `.csv` extension.

This validation step helps catch configuration or file errors early, preventing silent failures or misleading results later

In [29]:
# Create Dataset Directories if they don't exist
DATA_OUTPUT_DIR = Path("./Datasets/Outputs")
DATA_TEMP_DIR = Path("./Datasets/TempDatasets")

for directory in [DATA_OUTPUT_DIR, DATA_TEMP_DIR]:
        directory.mkdir(parents=True, exist_ok=True)

# Initialise Input Folder and Input Dataset Paths
DATA_INPUT_DIR = Path("./Datasets/Inputs")
FSA_PATH = DATA_INPUT_DIR / "FSA_data.csv"

def validate_csv_path(path: Path, label: str):
    if not path.exists():
        raise FileNotFoundError(f"{label} not found: {path.resolve()}")
    if not path.is_file():
        raise ValueError(f"{label} exists but is not a file: {path.resolve()}")
    if path.suffix.lower() != ".csv":
        raise ValueError(f"{label} is not a CSV file: {path.resolve()}")
    print(f"[OK] {label}: {path.resolve()}")
    
validate_csv_path(FSA_PATH, "FSA Data")

[OK] FSA Data: C:\Users\david\Desktop\GitHub\ICS5111-MiningLargeScaleData_Project\Datasets\Inputs\FSA_data.csv


## 2. Data Processing

This section defines the core text preprocessing function used throughout the data processing stage of the pipeline.

An English stopword list is first initialised using the NLTK library. Stopwords are common function words (such as “the”, “and”, or “is”) that typically carry little discriminative value in term-based retrieval models and are therefore removed during sparse preprocessing.

A preprocessing function is then defined to normalise and filter text data. The function performs basic cleaning by converting text to lowercase, tokenising it into individual terms, removing stopwords, and discarding non-alphabetic tokens. This reduces noise while preserving content-bearing terms that are informative for sparse retrieval techniques such as TF–IDF or BM25.

An optional case is provided to retain four digit numeric tokens corresponding to years. This is particularly relevant in financial and economic datasets, where temporal information can play an important role in retrieval and interpretation.

The output of this function is a string of filtered tokens, which is later stored alongside the original raw text. This helps us maintaining both representations and allows the pipeline to apply different retrieval strategies using the appropriate format of data.


In [30]:
# Get English stop words set
stop_words = set(stopwords.words('english'))
    
# Function to preprocess text for sparse representation
def tokenize_and_filter(text, keep_years=False):
    # Guard against NaN / non-string values
    if not isinstance(text, str):
        return ""
    
    # Lowercase, tokenize, remove stopwords and non-alphabetic tokens
    tokens = word_tokenize(text.lower())
    
    filtered_tokens = []
    
    # Filter tokens
    for word in tokens:
        # keep alphabetic words
        if word.isalpha() and word not in stop_words:
            filtered_tokens.append(word)
            
            # optionally keep years (4-digit numbers)
        elif keep_years == True and word.isdigit() and len(word) == 4:
            filtered_tokens.append(word)
        
    # Join tokens back into a single string
    return " ".join(filtered_tokens)

#### 2.1 [Kaggle Financial Sentiment Analysis Dataset](https://www.kaggle.com/sbhatti/financial-sentiment-analysis)

This subsection processes the Kaggle Financial Sentiment Analysis dataset. The dataset is first loaded from disk and annotated with a source identifier to track its origin once multiple datasets are combined.

Each sentence is then preprocessed using the shared preprocessing function, producing a cleaned `processed_text` representation while preserving the original raw text. This dual representation allows different retrieval strategies to operate on the most appropriate form of the data.

A year extraction step is applied to the processed text by scanning for valid four-digit numeric tokens within a predefined range. When present, the extracted year is stored as structured metadata, enabling temporal filtering or analysis in later stages of the pipeline. This was mainly done since the 2nd dataset automatically returns a year column.

In [31]:
# Load dataset from disk
kaggle_dataset = pd.read_csv(FSA_PATH)

# Apply preprocessing to dataset for each sentence and reorder columns for better readability
kaggle_dataset["source"] = "kaggle"
kaggle_dataset["processed_text"] = kaggle_dataset["Sentence"].apply(tokenize_and_filter, keep_years=True)

# Create a year column based on processed text (if any year exists in the text)
def extract_year(text, min_year=2000, max_year=2025):
    tokens = text.split()
    
    for token in tokens:
        # Check if token is a 4-digit number
        if token.isdigit() and len(token) == 4:
            # Convert token to integer
            year = int(token)
            
            # Check if year is within valid range
            if min_year <= year <= max_year:
                return year
        
    return None

# Extract year from processed text
kaggle_dataset["year"] = kaggle_dataset["processed_text"].apply(extract_year)

# Rename columns and reorder for better readability
kaggle_dataset = kaggle_dataset.rename(columns={"Sentence": "text", "Sentiment": "sentiment"})
kaggle_dataset = kaggle_dataset[["source", "text", "processed_text", "year", "sentiment"]]

print("Kaggle FSA Processed Data Sample:")
display(kaggle_dataset.head(10))

# Save processed dataset to disk
KAGGLE_OUTPUT_PATH = DATA_TEMP_DIR / "kaggle_fsa_dataset.csv"
kaggle_dataset.to_csv(KAGGLE_OUTPUT_PATH, index=False)

Kaggle FSA Processed Data Sample:


,source,text,processed_text,year,sentiment
0,kaggle,The GeoSolutions technology will leverage Bene...,geosolutions technology leverage benefon gps s...,NaN,positive
1,kaggle,"$ESI on lows, down $1.50 to $2.50 BK a real po...",esi lows bk real possibility,NaN,negative
2,kaggle,"For the last quarter of 2010 , Componenta 's n...",last quarter 2010 componenta net sales doubled...,2010.0,positive
3,kaggle,According to the Finnish-Russian Chamber of Co...,according chamber commerce major construction ...,NaN,neutral
4,kaggle,The Swedish buyout firm has sold its remaining...,swedish buyout firm sold remaining percent sta...,NaN,neutral
5,kaggle,$SPY wouldn't be surprised to see a green close,spy would surprised see green close,NaN,positive
6,kaggle,Shell's $70 Billion BG Deal Meets Shareholder ...,shell billion bg deal meets shareholder skepti...,NaN,negative
7,kaggle,SSH COMMUNICATIONS SECURITY CORP STOCK EXCHANG...,ssh communications security corp stock exchang...,2008.0,negative
8,kaggle,Kone 's net sales rose by some 14 % year-on-ye...,kone net sales rose first nine months 2008,2008.0,positive
9,kaggle,The Stockmann department store will have a tot...,stockmann department store total floor space s...,NaN,neutral


### 2.2 [World Bank Open Data](https://data.worldbank.org/)

This subsection retrieves structured economic indicators from the World Bank Open Data API. A base API endpoint is defined and used to fetch indicator values for selected countries over a specified time range.

A helper function is implemented to handle data retrieval. For each country, the function creates the appropriate API request, specifies the desired date range, and requests the data in JSON format. Error handling is applied to detect and halt execution in the case of invalid or failed requests.

Returned records are then parsed and transformed into a tabular format containing the country name, year, and indicator value. Requests are issued sequentially with a short delay between calls to avoid exceeding API rate limits. The output is a unified DataFrame containing all retrieved records.

In [32]:
WB_API_URL = "https://api.worldbank.org/v2/country/{country}/indicator/{indicator}"

# Function to fetch data from World Bank API
def fetch_api_data(indicator, countries, start_year, end_year, per_page=2000, rate_limit_delay=0.3):
    '''
    This fuction fetches data from World Bank API for given indicator and countries within specified year range.
    
    Parameters:
    - indicator (str): World Bank indicator code
    - contries (list): List of country codes
    - start_year (int): Start year for data
    - end_year (int): End year for data
    - per_page (int): Number of records per page
    - rate_limit_delay (float): Delay between requests to avoid rate limiting
    
    Returns:
    - DataFrame with columns ['country', 'year', 'value']
    '''
    all_records = []
    
    # Loop through countries one by one
    for country in countries:
        # Construct API URL
        url = WB_API_URL.format(
            country=country,
            indicator=indicator,
        )
        
        # Set query parameters
        params = {
            "date": f"{start_year}:{end_year}",
            "format": "json",
            "per_page": per_page,
        }
        
        # Make API request
        response = requests.get(url, params=params, timeout=60)
        response.raise_for_status() # Raise error for bad responses
        data = response.json()
    
        if len(data) < 2 or not data[1]:
            # No returned data for this country, skip
            time.sleep(rate_limit_delay)
            continue
        
        # Append records for this country
        for entry in data[1]:
            all_records.append({
                "country": entry["country"]["value"],
                "year": entry["date"],
                "value": entry["value"]
            })
    
        # Wait before next request so we don't spam the API
        time.sleep(rate_limit_delay)
    
    return pd.DataFrame(all_records)

This subsection retrieves economic indicators from the World Bank Open Data API for a predefined set of countries and years. A small collection of indicators relevant to financial and economic analysis is specified, along with the target countries and time range.

For each indicator, data is fetched for all countries using the previously defined API retrieval function. The results are annotated with a descriptive indicator name and combined into a single dataset covering multiple economic dimensions. The dataset is then labelled with a source identifier and its columns are reordered into a consistent structure.

> **[!NOTE]**
> **The API connection can occasionally timeout. If this happens please:**<br>
> **Clear All Outputs -> Restart Jupyter Kernel and Run All Again**<br> 
> **Thank you**  

In [33]:
# Congifure parameters for data fetching
COUNTRIES = {
    "MLT": "Malta",
    "ITA": "Italy",
    "DEU": "Germany",
    "FRA": "France",
    "ESP": "Spain",
    "PRT": "Portugal",
    "IRL": "Ireland",
    "NLD": "Netherlands",
}

INDICATORS = {
    "NY.GDP.MKTP.KD.ZG": "GDP growth (%)",
    "FP.CPI.TOTL.ZG": "Inflation rate (annual %)",
    "GC.DOD.TOTL.GD.ZS": "Government debt (% of GDP)",
    "SL.EMP.TOTL.ZS": "Employment rate (% of total labor force)",
    "SL.UEM.TOTL.ZS": "Unemployment rate (% of total labor force)"
}

START_YEAR = 2000
END_YEAR = 2025

# Fetch data
wb_dataset = []

# For each indicator, fetch data and append to dataset
for indicator_code, indicator_name in INDICATORS.items():
    df = fetch_api_data(
        indicator=indicator_code,
        countries=list(COUNTRIES.keys()),
        start_year=START_YEAR,
        end_year=END_YEAR
        # per_page = 2000,          # Parameters already set to default values in function
        # rate_limit_delay = 0.3    # Parameters already set to default values in function
    )

    df["indicator_name"] = indicator_name
    wb_dataset.append(df)

# Combine all indicator data into a single DataFrame
wb_dataset = pd.concat(wb_dataset, ignore_index=True)
wb_dataset["source"] = "wb_open_data"

# Reorder columns for better readability
wb_dataset = wb_dataset[["source", "country", "year", "indicator_name", "value"]]

print("World Bank Economic Indicators Data Sample:")
display(wb_dataset.head(10))

World Bank Economic Indicators Data Sample:


,source,country,year,indicator_name,value
0,wb_open_data,Malta,2024,GDP growth (%),6.799903
1,wb_open_data,Malta,2023,GDP growth (%),10.621519
2,wb_open_data,Malta,2022,GDP growth (%),2.483100
3,wb_open_data,Malta,2021,GDP growth (%),13.411704
4,wb_open_data,Malta,2020,GDP growth (%),-3.457283
5,wb_open_data,Malta,2019,GDP growth (%),4.085056
6,wb_open_data,Malta,2018,GDP growth (%),7.189215
7,wb_open_data,Malta,2017,GDP growth (%),12.971342
8,wb_open_data,Malta,2016,GDP growth (%),4.078004
9,wb_open_data,Malta,2015,GDP growth (%),9.620703


### 2.3 Formatting World Bank API Output

Since retrieval models operate on text rather than raw numerical tables, each World Bank data row is converted into a natural-language sentence.

For example:
> **“In 2024, Malta’s GDP growth (%) was 6.79.”**

This transformation allows for structured indicators to be indexed and retrieved in the same way as unstructured text documents.

In [34]:
# Create natural language representation for each World Bank data row
def wb_row_to_text(row):
    return f"In {row['year']}, {row['country']}'s {row['indicator_name']} was {row['value']:.6f}."

wb_dataset["text"] = wb_dataset.apply(wb_row_to_text, axis=1)
display(wb_dataset.head(10))

,source,country,year,indicator_name,value,text
0,wb_open_data,Malta,2024,GDP growth (%),6.799903,"In 2024, Malta's GDP growth (%) was 6.799903."
1,wb_open_data,Malta,2023,GDP growth (%),10.621519,"In 2023, Malta's GDP growth (%) was 10.621519."
2,wb_open_data,Malta,2022,GDP growth (%),2.483100,"In 2022, Malta's GDP growth (%) was 2.483100."
3,wb_open_data,Malta,2021,GDP growth (%),13.411704,"In 2021, Malta's GDP growth (%) was 13.411704."
4,wb_open_data,Malta,2020,GDP growth (%),-3.457283,"In 2020, Malta's GDP growth (%) was -3.457283."
5,wb_open_data,Malta,2019,GDP growth (%),4.085056,"In 2019, Malta's GDP growth (%) was 4.085056."
6,wb_open_data,Malta,2018,GDP growth (%),7.189215,"In 2018, Malta's GDP growth (%) was 7.189215."
7,wb_open_data,Malta,2017,GDP growth (%),12.971342,"In 2017, Malta's GDP growth (%) was 12.971342."
8,wb_open_data,Malta,2016,GDP growth (%),4.078004,"In 2016, Malta's GDP growth (%) was 4.078004."
9,wb_open_data,Malta,2015,GDP growth (%),9.620703,"In 2015, Malta's GDP growth (%) was 9.620703."


The generated natural language sentences are further processed using the same preprocessing function used for the Kaggle dataset.

In this case, year tokens are retained to support time-specific queries (e.g., “GDP growth in 2020”). This produces a sparse representation suitable for keyword-based retrieval while preventing the loss of temporal information.

In [35]:
wb_dataset["processed_text"] = wb_dataset["text"].apply(tokenize_and_filter, keep_years=True)
display(wb_dataset.head(10))

# Save fetched data to CSV
WBF_OUTPUT_PATH = DATA_TEMP_DIR / "wb_economic_indicators_dataset.csv"
wb_dataset.to_csv(WBF_OUTPUT_PATH, index=False)

,source,country,year,indicator_name,value,text,processed_text
0,wb_open_data,Malta,2024,GDP growth (%),6.799903,"In 2024, Malta's GDP growth (%) was 6.799903.",2024 malta gdp growth
1,wb_open_data,Malta,2023,GDP growth (%),10.621519,"In 2023, Malta's GDP growth (%) was 10.621519.",2023 malta gdp growth
2,wb_open_data,Malta,2022,GDP growth (%),2.483100,"In 2022, Malta's GDP growth (%) was 2.483100.",2022 malta gdp growth
3,wb_open_data,Malta,2021,GDP growth (%),13.411704,"In 2021, Malta's GDP growth (%) was 13.411704.",2021 malta gdp growth
4,wb_open_data,Malta,2020,GDP growth (%),-3.457283,"In 2020, Malta's GDP growth (%) was -3.457283.",2020 malta gdp growth
5,wb_open_data,Malta,2019,GDP growth (%),4.085056,"In 2019, Malta's GDP growth (%) was 4.085056.",2019 malta gdp growth
6,wb_open_data,Malta,2018,GDP growth (%),7.189215,"In 2018, Malta's GDP growth (%) was 7.189215.",2018 malta gdp growth
7,wb_open_data,Malta,2017,GDP growth (%),12.971342,"In 2017, Malta's GDP growth (%) was 12.971342.",2017 malta gdp growth
8,wb_open_data,Malta,2016,GDP growth (%),4.078004,"In 2016, Malta's GDP growth (%) was 4.078004.",2016 malta gdp growth
9,wb_open_data,Malta,2015,GDP growth (%),9.620703,"In 2015, Malta's GDP growth (%) was 9.620703.",2015 malta gdp growth


### 2.4 Combining Datasets

In [36]:
print("Kaggle FSA Processed Data Sample:")
display(kaggle_dataset.head())
print("World Bank Economic Indicators Data Sample:")
display(wb_dataset.head())

Kaggle FSA Processed Data Sample:


,source,text,processed_text,year,sentiment
0,kaggle,The GeoSolutions technology will leverage Bene...,geosolutions technology leverage benefon gps s...,NaN,positive
1,kaggle,"$ESI on lows, down $1.50 to $2.50 BK a real po...",esi lows bk real possibility,NaN,negative
2,kaggle,"For the last quarter of 2010 , Componenta 's n...",last quarter 2010 componenta net sales doubled...,2010.0,positive
3,kaggle,According to the Finnish-Russian Chamber of Co...,according chamber commerce major construction ...,NaN,neutral
4,kaggle,The Swedish buyout firm has sold its remaining...,swedish buyout firm sold remaining percent sta...,NaN,neutral


World Bank Economic Indicators Data Sample:


,source,country,year,indicator_name,value,text,processed_text
0,wb_open_data,Malta,2024,GDP growth (%),6.799903,"In 2024, Malta's GDP growth (%) was 6.799903.",2024 malta gdp growth
1,wb_open_data,Malta,2023,GDP growth (%),10.621519,"In 2023, Malta's GDP growth (%) was 10.621519.",2023 malta gdp growth
2,wb_open_data,Malta,2022,GDP growth (%),2.483100,"In 2022, Malta's GDP growth (%) was 2.483100.",2022 malta gdp growth
3,wb_open_data,Malta,2021,GDP growth (%),13.411704,"In 2021, Malta's GDP growth (%) was 13.411704.",2021 malta gdp growth
4,wb_open_data,Malta,2020,GDP growth (%),-3.457283,"In 2020, Malta's GDP growth (%) was -3.457283.",2020 malta gdp growth


Before merging the datasets, empty metadata fields are added to the Kaggle dataset to ensure schema consistency across both data sources.

Both datasets are then reordered to follow a shared column structure:

- `source`
- `text`
- `processed_text`
- `country`
- `year`
- `indicator_name`
- `value`

The aligned datasets are concatenated into a single unified table, forming the final dataset used throughout the RAG pipeline. A sample of the merged data is displayed to verify correct integration.

The resulting dataset is saved to disk as a CSV file and represents the complete RAG corpus for this project, combining unstructured textual data with structured economic metadata from multiple sources.


In [37]:
# In order to keep metadata consistent across both datasets, we add empty columns to the Kaggle dataset
kaggle_dataset["country"] = None
kaggle_dataset["indicator_name"] = None
kaggle_dataset["value"] = None

# Final reordering of columns
kaggle_dataset = kaggle_dataset[
    ["source", "text", "processed_text", "country", "year", "indicator_name", "value"]
]

wb_dataset = wb_dataset[
    ["source", "text", "processed_text", "country", "year", "indicator_name", "value"]
]

# Combine both datasets into final RAG dataset
final_rag_dataset = pd.concat([kaggle_dataset, wb_dataset], ignore_index=True)

C:\Users\david\AppData\Local\Temp\ipykernel_28160\2366872830.py:16: FutureWarning:

The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.



### 2.5 Handling Duplicate Data

In [38]:
def remove_duplicate_rows(df, subset=None, keep="first"):
    """
    Removes duplicate rows from a DataFrame and reports how many were removed.

    Parameters:
    - df (pd.DataFrame): Input DataFrame
    - subset (list or None): Columns to consider for duplicate detection.
                             If None, all columns are used.
    - keep (str): Which duplicate to keep ('first', 'last', or False)

    Returns:
    - cleaned_df (pd.DataFrame): DataFrame with duplicates removed
    - removed_count (int): Number of rows removed
    """

    original_count = len(df)

    cleaned_df = df.drop_duplicates(subset=subset, keep=keep).reset_index(drop=True)

    removed_count = original_count - len(cleaned_df)
    
    return cleaned_df, removed_count

In [39]:
# Remove duplicate rows from final RAG dataset
final_rag_dataset, removed_rows_count = remove_duplicate_rows(final_rag_dataset)
print(f"Removed {removed_rows_count} duplicate rows from final RAG dataset.\n")

print("Final RAG Dataset Sample:")
display(final_rag_dataset.head(20))

# Save final RAG dataset to CSV
RAG_OUTPUT_PATH = DATA_OUTPUT_DIR / "final_rag_dataset.csv"
final_rag_dataset.to_csv(RAG_OUTPUT_PATH, index=False)

Removed 520 duplicate rows from final RAG dataset.

Final RAG Dataset Sample:


,source,text,processed_text,country,year,indicator_name,value
0,kaggle,The GeoSolutions technology will leverage Bene...,geosolutions technology leverage benefon gps s...,None,NaN,None,NaN
1,kaggle,"$ESI on lows, down $1.50 to $2.50 BK a real po...",esi lows bk real possibility,None,NaN,None,NaN
2,kaggle,"For the last quarter of 2010 , Componenta 's n...",last quarter 2010 componenta net sales doubled...,None,2010.0,None,NaN
3,kaggle,According to the Finnish-Russian Chamber of Co...,according chamber commerce major construction ...,None,NaN,None,NaN
4,kaggle,The Swedish buyout firm has sold its remaining...,swedish buyout firm sold remaining percent sta...,None,NaN,None,NaN
5,kaggle,$SPY wouldn't be surprised to see a green close,spy would surprised see green close,None,NaN,None,NaN
6,kaggle,Shell's $70 Billion BG Deal Meets Shareholder ...,shell billion bg deal meets shareholder skepti...,None,NaN,None,NaN
7,kaggle,SSH COMMUNICATIONS SECURITY CORP STOCK EXCHANG...,ssh communications security corp stock exchang...,None,2008.0,None,NaN
8,kaggle,Kone 's net sales rose by some 14 % year-on-ye...,kone net sales rose first nine months 2008,None,2008.0,None,NaN
9,kaggle,The Stockmann department store will have a tot...,stockmann department store total floor space s...,None,NaN,None,NaN


## 3. Simple EDA

This section provides an exploratory analysis of the final RAG dataset, with the main aim being to understand the structure and contents of the dataset before applying retrieval and generation methods.

### 3.1 Distribution of Data Sources

This subsection looks at the distribution of records across the different data sources included in the final dataset. A histogram is used to visualise the relative contribution of each source, helping to identify any potential imbalance between sources.

In [40]:
fig = px.histogram(
    final_rag_dataset,
    x="source",
    color="source",
    text_auto=True,
    title="Distribution of Data Sources",
    labels={"source": "Data Source"},
    color_discrete_sequence=["#AEC6CF", "#FF6961"]  # pastel blue & pastel pink
)

fig.show()

### 3.2 Missing Values per Column

This subsection analyses missing values across all columns in the dataset. The total number of missing entries per column is calculated and visualised to highlight which fields are sparsely populated.

This analysis helps distinguish between expected missing values (e.g. structured metadata not applicable to all records) and potential data quality issues that may require attention in later stages.

In [41]:
missing_df = final_rag_dataset.isna().sum().reset_index()
missing_df = missing_df[missing_df[0] > 0].sort_values(by=0, ascending=True)

if not missing_df.empty:
    missing_df.columns = ["column", "missing_count"]

    fig = px.bar(
        missing_df,
        x="column",
        y="missing_count",
        text_auto=True,
        title="Missing Values per Column",
        labels={"missing_count": "Number of Missing Values"},
        color="column",
        color_discrete_sequence=[
            "#AEC6CF",  # pastel blue
            "#FF6961",  # pastel pink
            "#C1E1C1",  # pastel green
            "#D7BDE2",  # pastel purple
            "#FFF2CC",  # pastel yellow
            "#FADADD",  # pastel peach
            "#D7B4F3"   # pastel lavender
        ]
    )

    fig.show()

## 4. Sparse Retrieval Methods

### BM25

In [42]:
final_rag_dataset = pd.read_csv('Datasets/Outputs/final_rag_dataset.csv')
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

#Calculating BM25 Score

def termFrequency(term, document, avgDocLength, k, b):
    termCount = document.count(term)
    docLength = len(document)
    
    return (termCount / (termCount + k*(1 - b + b * (docLength / avgDocLength))))

def inverseDocFrequency(term, totalCorpLength):
    docsContainingTerm = final_rag_dataset['processed_text'].str.contains(term).sum()

    return math.log((totalCorpLength - docsContainingTerm + 0.5) / (docsContainingTerm + 0.5))

def bm25Score(query, document, avgDocLength, totalCorpLength, k, b):
    bmScore = 0
    for term in query:
        tf = termFrequency(term, document, avgDocLength, k, b)
        idf = inverseDocFrequency(term, totalCorpLength)
        bmScore += tf*idf
    return bmScore

#k controls term frequency scaling
#b controls document length normalization
def bm25_retrieval(query, top_k=5, k=1.2, b=0.75):
    avgDocLength = final_rag_dataset['processed_text'].str.split().str.len().mean()
    totalCorpLength = len(final_rag_dataset)
    results = []

    cleanedQuery = ''.join(char for char in query if char.isalnum() or char.isspace())
    processedQuery = tokenizer.tokenize(cleanedQuery.lower())

    for doc in final_rag_dataset.itertuples():
        processedDoc = doc.processed_text.split()
        score = bm25Score(processedQuery, processedDoc, avgDocLength, totalCorpLength, k, b)
        if score != 0 and (doc.processed_text, score) not in results:
            results.append({
                "text": doc.text,
                "source": doc.source,
                "year": doc.year,
                "bm25_score": score
            })

    #Ordering by relevance
    results = pd.DataFrame(results)
    topK = (
        results
        .sort_values(by="bm25_score", ascending=False)
        .head(top_k)
        .reset_index(drop=True)
    )

    topK["rank"] = topK.index + 1
    
    return topK


## 5. Dense Retrieval Methods

This section, impliments two dense retrieval approaches for information retrieval in a RAG pipeline. For sentence-transformers, the same feature space is used for both the database and user queries. While for Dense Passage Retrieval (DPR) separate neural encoders are used for queries and documents to perform semantic retrieval.

### 5.1 Sentence-Transformers

In this section, a check is performed to ensure that the dataset contains a `text` column, which represents the natural language content of each document. This column is essential, as it is the information that will later be encoded and retrieved by the models. Document texts are extracted into a list (`doc_texts`). This list serves as the input for the embedding models used in dense retrieval

In [43]:
# Make sure 'text' column exists
if not "text" in final_rag_dataset.columns:
    raise ValueError("!!! Text column not found in RAG dataset !!!")

# Extract document texts for encoding
documents = final_rag_dataset.sort_index().reset_index()
doc_texts = documents["text"].tolist()

display(documents.head())

,index,source,text,processed_text,country,year,indicator_name,value
0,0,kaggle,The GeoSolutions technology will leverage Bene...,geosolutions technology leverage benefon gps s...,NaN,NaN,NaN,NaN
1,1,kaggle,"$ESI on lows, down $1.50 to $2.50 BK a real po...",esi lows bk real possibility,NaN,NaN,NaN,NaN
2,2,kaggle,"For the last quarter of 2010 , Componenta 's n...",last quarter 2010 componenta net sales doubled...,NaN,2010.0,NaN,NaN
3,3,kaggle,According to the Finnish-Russian Chamber of Co...,according chamber commerce major construction ...,NaN,NaN,NaN,NaN
4,4,kaggle,The Swedish buyout firm has sold its remaining...,swedish buyout firm sold remaining percent sta...,NaN,NaN,NaN,NaN


To enable dense semantic retrieval, all documents in the dataset are converted into vector representations using a Sentence-Transformers model.

The model used is **all-MiniLM-L6-v2**, a lightweight transformer trained to produce semantically meaningful sentence embeddings. This allows documents with similar meaning to be located even when they do not share the same keywords. Each document text is passed through the model and encoded into a fixed-size numerical vector. The embeddings are normalised so that similarity comparisons depend only on semantic direction rather than feature space.

In [44]:
# Load SentenceTransformer model
# https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2
st_model  = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

# Encode all document texts into normalized embeddings
doc_embeddings_st = st_model.encode(
    doc_texts,
    batch_size=16,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
)

Batches: 100%|██████████| 383/383 [00:21<00:00, 17.76it/s]


This function implements dense document retrieval using the previously generated Sentence-Transformer embeddings. User queries are encoded into a vector representation using the same model and normalisation settings as the document embeddings. This ensures that both queries and documents lie in the same semantic vector space, allowing meaningful similarity comparisons.

Cosine similarity is then computed between the query embedding and all document embeddings. This measures how close each document is to the query in terms of semantic meaning rather than surface-level keyword overlap. The documents are ranked based on their similarity scores, and the top-k most relevant documents are selected. These results are returned in a structured table containing the document text, metadata, similarity score, and rank.

In [45]:
def st_retrieval(query, top_k = 5, model= st_model):
    # Encode and normalize query embedding (same settings as corpus embeddings)
    query_embedding = model.encode(
        query,
        convert_to_numpy=True,
        normalize_embeddings=True
    )

    # Compute cosine similarity between query and all document embeddings
    scores = cosine_similarity(query_embedding.reshape(1, -1), doc_embeddings_st)[0]
    scores = np.round((scores * 100), 3)

    # Select top-k document indices based on similarity
    top_indices = np.argsort(scores)[::-1][:top_k]

    # Build a result table for inspection
    results = documents.iloc[top_indices][["text", "source", "year"]].copy()
    results["similarity_score"] = scores[top_indices]
    results["rank"] = np.arange(1, len(results) + 1)

    return results.reset_index(drop=True)

### 5.2 DPR

This section sets up the Dense Passage Retrieval using dual-encoder architecture. Unlike Sentence-Transformers, which use a single model to embed both queries and documents into the same vector space, DPR employs two separate encoders:
- A **question encoder**, used to represent user queries, and  
- A **context encoder**, used to represent documents in the corpus.

Both encoders are pretrained on large-scale question answering datasets, enabling them to map semantically related queries and documents close to each other in the embedding space. This separation allows DPR to specialise each encoder for its respective role, making it particularly effective for question-answering style retrieval tasks.


In [46]:
# Question encoder (for queries)
# https://huggingface.co/facebook/dpr-question_encoder-single-nq-base
q_tokenizer = DPRQuestionEncoderTokenizer.from_pretrained("facebook/dpr-question_encoder-single-nq-base")
q_encoder = DPRQuestionEncoder.from_pretrained("facebook/dpr-question_encoder-single-nq-base")

# Context encoder (for documents)
# https://huggingface.co/facebook/dpr-ctx_encoder-single-nq-base
ctx_tokenizer = DPRContextEncoderTokenizer.from_pretrained("facebook/dpr-ctx_encoder-single-nq-base")
ctx_encoder = DPRContextEncoder.from_pretrained("facebook/dpr-ctx_encoder-single-nq-base" )

Some weights of the model checkpoint at facebook/dpr-question_encoder-single-nq-base were not used when initializing DPRQuestionEncoder: ['question_encoder.bert_model.pooler.dense.bias', 'question_encoder.bert_model.pooler.dense.weight']
- This IS expected if you are initializing DPRQuestionEncoder from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing DPRQuestionEncoder from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
The tokenizer class you load from this checkpoint is not the same type as the class this function is called from. It may result in unexpected tokenization. 
The tokenizer class you load from this checkpoint is 'DPRQuestionEncoderTokenizer'. 
The class this function is called from is 'DPRCon

This function encodes all documents, transforming each document into a dense vector representation. To improve efficiency and avoid memory issues, the documents are processed in batches. Each batch is tokenized and passed through the context encoder to obtain its embedding and the resulting embeddings represent the semantic content of each document and are concatenated into a single matrix, which forms the dense index used later for similarity-based retrieval. This step prepares the document collection for fast and accurate dense retrieval within the DPR-based component of the RAG pipeline

In [47]:
def encode_documents_dpr(texts, batch_size=16):
    # List to store embeddings for all document batches
    all_embeddings = []

    # Process documents in batches to control memory usage
    for i in range(0, len(texts), batch_size):
        # Select a batch
        batch = texts[i:i + batch_size]

        # Tokenize the batch using the DPR context tokenizer
        inputs = ctx_tokenizer(
            batch,
            padding=True,
            truncation=True,
            max_length=256,
            return_tensors="pt"
        )

        # Obtain embeddings from the DPR context encoder
        outputs = ctx_encoder(**inputs)
        # Store embeddings of the current batch
        all_embeddings.append(outputs.pooler_output.detach().numpy())

    # Concatenate all batch embeddings into a single tensor
    return np.vstack(all_embeddings)

After encoding all documents into dense vector representations, the embeddings are indexed using FAISS. An index is created, which is well-suited for DPR since both query and document embeddings are compared using dot-product similarity. This allows fast retrieval of the most relevant documents without having to compute similarities against the entire corpus at query time.

By adding all document embeddings to the FAISS index, the system can perform scalable and efficient dense retrieval, making it suitable for larger datasets and real-time query processing. This indexing step allows DPR to operate as an efficient retrieval component within the RAG pipeline.

In [48]:
doc_embeddings_dpr = encode_documents_dpr(doc_texts)

faiss_index = faiss.IndexFlatIP(doc_embeddings_dpr.shape[1])
faiss_index.add(doc_embeddings_dpr)

This function performs dense document retrieval using the DPR question encoder and the FAISS index. First, the user query is tokenized and encoded into a dense vector representation using the DPR question encoder. This vector captures the semantic meaning of the query in a form that can be directly compared with document embeddings.

The query embedding is then searched against the FAISS index, which efficiently identifies the top-k most similar document vectors using inner product similarity. This avoids performing a brute-force comparison against the entire dataset.

In [49]:
def dpr_retrieval(query, top_k=5):
    # Tokenize the query for the DPR question encoder
    query_inputs  = q_tokenizer(
        query,
        truncation=True,
        max_length=64,
        return_tensors="pt"
    )

    # Encode the query into a dense vector representation
    query_embedding = q_encoder(**query_inputs).pooler_output.detach().numpy()

    # Search the FAISS index for top-k similar documents
    scores, indices = faiss_index.search(query_embedding, top_k)

    # Build a results table with relevant document information
    results = documents.iloc[indices[0]][["text", "source", "year"]].copy()
    results["similarity_score"] = scores[0]
    results["rank"] = np.arange(1, len(results) + 1)

    return results.reset_index(drop=True)

## 6. Comparative Evaluation

In [52]:
# Compute BLEU score using SacreBLEU
def bleu_eval(all_candidates, all_references):
    """
    Computes corpus-level BLEU score using SacreBLEU
    and returns both the score and the evaluation signature.
    """

    # SacreBLEU expects:
    # candidates = list of generated texts
    # references = list of reference lists (one list per reference set)
    references = [all_references]

    # Compute corpus BLEU
    bleu_result = corpus_bleu(all_candidates, references)

    # Extract BLEU score
    bleu_score = bleu_result.score

    # Extract the signature (how BLEU was computed)
    #bleu_signature = bleu_result.signature

    return bleu_score

# Compute ROUGE scores using rouge_score
def rouge_eval(candidate, reference):
    # Create a scorer for ROUGE-1 and ROUGE-L
    scorer = rouge_scorer.RougeScorer(['rouge1', 'rougeL'], use_stemmer=True)

    # Compute the scores between reference and candidate
    raw_scores  = scorer.score(reference, candidate)

    # Extract F1 scores, Precision, Recall
    formated_scores = {
        "rouge1_f1": raw_scores['rouge1'].fmeasure,
        "rouge1_precision": raw_scores['rouge1'].precision,
        "rouge1_recall": raw_scores['rouge1'].recall,
        "rougeL_f1": raw_scores['rougeL'].fmeasure,
        "rougeL_precision": raw_scores['rougeL'].precision,
        "rougeL_recall": raw_scores['rougeL'].recall,
    }
    
    return formated_scores

### 6.1 RAG Answer Generation

This subsection implements a barebones QA interface that can operate with different retrieval methods. Instead of being tied to a specific retriever, the function accepts a retrieval function as input. This allows the same generation logic to be reused with different retrieval strategies, such as Sentence-Transformers, DPR, or BM25. For a given query, the selected retriever is first used to obtain the top-k most relevant documents. 

The texts of these documents are then concatenated to form a context, which is combined with the original question to construct a prompt for the generative model. A pretrained sequence-to-sequence model (FLAN-T5) is used to generate a natural language answer grounded in the retrieved context. This ensures that the responses are based on the document collection rather than solely on the model’s internal knowledge.

In [53]:
gen_tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-large")
gen_model = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-large")

def rag_generate(f, query, top_k=5):
    results = f(query, top_k)
    context = " ".join(results["text"].tolist())

    prompt = f"""
    Context: {context}

    Question: {query}

    Answer the question in one complete sentence:
    """

    # Tokenize for generator
    inputs = gen_tokenizer(prompt, return_tensors="pt", truncation=True)
    # Generate answer
    outputs = gen_model.generate(**inputs, max_new_tokens=100)
    # Isolate answer
    answer = gen_tokenizer.decode(outputs[0], skip_special_tokens=True)

    # return answer with top similarity_score and top rank
    return answer, results

To evaluate the effect of different retrieval strategies on answer generation, the RAG pipeline is tested using multiple retrievers under the same generative model. With a list of queries for each retriever

In [55]:
queries = [
    "How did inflation change in Italy after 2020?",
    "Investors are worried about rising inflation amid political uncertainty", # Augmented
    "Investors are worried about inflation in politics", # Originator of Augmented
    "What happened to unemployment in Malta after COVID?",
    "How did GDP change in European countries after the financial crisis?",
    "What trends are observed in European interest rates in recent years?",
    "How has economic growth evolved in the Eurozone after 2015?"
]

ideal_responses = [
    # 1. Inflation in Italy after 2020
    "After 2020, inflation in Italy rose sharply.",
    # 2. Augmented inflation + political uncertainty
    "Rising inflation combined with political uncertainty has increased investor concerns.",
    # 3. Original inflation + politics
    "Rising inflation combined with political uncertainty has increased investor concerns.",
    # 4. Unemployment in Malta after COVID
    "Following the COVID-19 pandemic, unemployment in Malta increased initially but gradually declined.",
    # 5. GDP after the financial crisis
    "After the financial crisis, GDP in many European countries declined sharply.",
    # 6. European interest rate trends
    "European interest rates remained very low for an extended period before rising.",
    # 7. Eurozone growth after 2015
    "Economic growth in the Eurozone improved moderately after 2015."
]


retrievers = [bm25_retrieval, st_retrieval, dpr_retrieval]

for retriever in retrievers:
    print(f"{'-'*100}\n{retriever.__name__} RAG Results:\n")
    all_candidates = []
    all_references = []

    for i, q in enumerate(queries):
        answer, results = rag_generate(retriever, q, top_k=5)
        rogueScore = rouge_eval(answer, ideal_responses[i])

        # Collect for BLEU
        all_candidates.append(answer)
        all_references.append(ideal_responses[i])

        print(f"Q: {q}\nA: {answer}\nRogue Score: {rogueScore}\nResulting Table: \n")
        display(results)
        print("\n")

    bleu_score = bleu_eval(all_candidates, all_references)

    print("SacreBLEU Results")
    print(f"BLEU Score: {bleu_score:.2f}")

----------------------------------------------------------------------------------------------------
bm25_retrieval RAG Results:

Q: How did inflation change in Italy after 2020?
A: In 2020, Italy's Inflation rate (annual %) was -0.137708
Rogue Score: {'rouge1_f1': 0.47058823529411764, 'rouge1_precision': 0.4, 'rouge1_recall': 0.5714285714285714, 'rougeL_f1': 0.23529411764705882, 'rougeL_precision': 0.2, 'rougeL_recall': 0.2857142857142857}
Resulting Table: 



,text,source,year,bm25_score,rank
0,"In 2020, Italy's Inflation rate (annual %) was...",wb_open_data,2020.0,7.250883,1
1,"In 2020, Italy's GDP growth (%) was -8.868221.",wb_open_data,2020.0,5.591258,2
2,"In 2020, Italy's Government debt (% of GDP) wa...",wb_open_data,2020.0,5.054345,3
3,"In 2020, Netherlands's Inflation rate (annual ...",wb_open_data,2020.0,4.951948,4
4,"In 2020, Portugal's Inflation rate (annual %) ...",wb_open_data,2020.0,4.951948,5




Q: Investors are worried about rising inflation amid political uncertainty
A: None of the above choices .
Rogue Score: {'rouge1_f1': 0.0, 'rouge1_precision': 0.0, 'rouge1_recall': 0.0, 'rougeL_f1': 0.0, 'rougeL_precision': 0.0, 'rougeL_recall': 0.0}
Resulting Table: 



,text,source,year,bm25_score,rank
0,"At the moment , Valio is not worried , but if ...",kaggle,NaN,3.819421,1
1,Small investors have voiced fears that the sha...,kaggle,NaN,3.676256,2
2,Industry NewsWolseley confident in reslilience...,kaggle,NaN,3.665098,3
3,Tullow Oil Suspends Dividend Amid Oil Price Fall,kaggle,NaN,3.504542,4
4,Royal Mail share price rallies amid positive b...,kaggle,NaN,3.357462,5




Q: Investors are worried about inflation in politics
A: None of the above choices .
Rogue Score: {'rouge1_f1': 0.0, 'rouge1_precision': 0.0, 'rouge1_recall': 0.0, 'rougeL_f1': 0.0, 'rougeL_precision': 0.0, 'rougeL_recall': 0.0}
Resulting Table: 



...



Q: What trends are observed in European interest rates in recent years?
A: higher year-on-year
Rogue Score: {'rouge1_f1': 0.0, 'rouge1_precision': 0.0, 'rouge1_recall': 0.0, 'rougeL_f1': 0.0, 'rougeL_precision': 0.0, 'rougeL_recall': 0.0}
Resulting Table: 



,text,source,year,similarity_score,rank
0,Earnings per share for the quarter were also h...,kaggle,NaN,76.873749,1
1,The diluted loss per share narrowed to EUR 0.2...,kaggle,NaN,76.762032,2
2,Diluted earnings per share ( EPS ) rose to EUR...,kaggle,NaN,76.218689,3
3,Diluted earnings per share ( EPS ) declined to...,kaggle,NaN,76.045929,4
4,"European stocks hover near 3-week low, Dialog ...",kaggle,NaN,75.910233,5




Q: How has economic growth evolved in the Eurozone after 2015?
A: Strongest growth was seen in the new markets in Russia , the Czech Republic , and Slovakia .
Rogue Score: {'rouge1_f1': 0.25, 'rouge1_precision': 0.2, 'rouge1_recall': 0.3333333333333333, 'rougeL_f1': 0.25, 'rougeL_precision': 0.2, 'rougeL_recall': 0.3333333333333333}
Resulting Table: 



,text,source,year,similarity_score,rank
0,Strongest growth was seen in the new markets i...,kaggle,NaN,75.346146,1
1,"In 2023, Spain's GDP growth (%) was 2.460858.",wb_open_data,2023.0,73.443466,2
2,"In 2024, Spain's GDP growth (%) was 3.455254.",wb_open_data,2024.0,73.412933,3
3,The growth of net sales has continued favourab...,kaggle,NaN,73.400192,4
4,"The highest growth would be in China , while w...",kaggle,NaN,73.321808,5




SacreBLEU Results
BLEU Score: 1.49


---
> *This project was developed as part of the **ICS5111** course at the University of Malta.*